# Phase 4: Evaluation and Bias Audit
Run the cells in order. Step 1 asks you to pick `WESAD_ML_Ready_Features.csv` from your computer.

In [ ]:
from google.colab import files
up = files.upload()   # choose WESAD_ML_Ready_Features.csv
print(list(up.keys()))

Step 2: the evaluation script. It trains the models and takes a few minutes (LOSO runs 15 folds).

In [ ]:
%%writefile phase4_evaluation.py
"""
Phase 4 - Evaluation and Bias Audit (WESAD stress classification, Group F26-02)

Run in Colab or locally:  python phase4_evaluation.py WESAD_ML_Ready_Features.csv
Needs: pandas numpy scikit-learn matplotlib seaborn torch
Outputs (in ./phase4_outputs): CSV tables + PNG figures used for the report.

Sections
  A. Reproduce Phase 3 (random 80/20 split)        -> the "optimistic" number
  B. Subject-independent split (train S2-S13, test S14-S17)
  C. Leave-One-Subject-Out (LOSO) for RF, 1D-CNN, Logistic Regression
  D. ROC and Precision-Recall curves
  E. Bias / robustness checks: per-subject results, ECG-only vs EDA-only,
     per-subject normalisation
"""
import sys, os
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, confusion_matrix)
import torch, torch.nn as nn

CSV = sys.argv[1] if len(sys.argv) > 1 else "WESAD_ML_Ready_Features.csv"
OUT = "phase4_outputs"; os.makedirs(OUT, exist_ok=True)
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

d = pd.read_csv(CSV)
y = (d.State == "Stressed").astype(int).values
FEATS = [c for c in d.columns if c not in ("Subject", "State")]
ECG = [c for c in FEATS if c.startswith("ECG")]
EDA = [c for c in FEATS if c.startswith("EDA")]
subjects = sorted(d.Subject.unique(), key=lambda s: int(s[1:]))


# ---------------- models (same settings as Phase 3) ----------------
class StressCNN1D(nn.Module):
    def __init__(self, n_feats=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, 3, padding=1), nn.BatchNorm1d(16), nn.ReLU(),
            nn.Conv1d(16, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(32, 2))
    def forward(self, x): return self.net(x)


def fit_predict(kind, Xtr, ytr, Xte, epochs=20):
    """Return P(stressed) for Xte. Scaler is fitted on training data only."""
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    if kind == "RF":
        m = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=4,
                                   random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
        return m.predict_proba(Xte)[:, 1]
    if kind == "LR":
        return LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    if kind == "CNN":
        net = StressCNN1D(Xtr.shape[1]).to(DEV)
        opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
        lossf = nn.CrossEntropyLoss()
        Xt = torch.tensor(Xtr, dtype=torch.float32).unsqueeze(1)
        yt = torch.tensor(ytr, dtype=torch.long)
        for _ in range(epochs):
            net.train(); perm = torch.randperm(len(Xt))
            for i in range(0, len(Xt), 32):
                idx = perm[i:i + 32]
                opt.zero_grad()
                lossf(net(Xt[idx].to(DEV)), yt[idx].to(DEV)).backward()
                opt.step()
        net.eval()
        with torch.no_grad():
            logits = net(torch.tensor(Xte, dtype=torch.float32).unsqueeze(1).to(DEV))
            return torch.softmax(logits, 1)[:, 1].cpu().numpy()


def metrics(yt, pr):
    p = (pr >= 0.5).astype(int)
    out = dict(acc=accuracy_score(yt, p), f1=f1_score(yt, p), auc=roc_auc_score(yt, pr),
               ap=average_precision_score(yt, pr))
    tn, fp, fn, tp = confusion_matrix(yt, p, labels=[0, 1]).ravel()
    out.update(tn=tn, fp=fp, fn=fn, tp=tp)
    return out


rows = []

# ---------------- A. random split (Phase 3 reproduction) ----------------
Xtr, Xte, ytr, yte = train_test_split(d[FEATS].values, y, test_size=0.2, stratify=y, random_state=SEED)
for k in ["RF", "CNN"]:
    rows.append(dict(protocol="random_80_20", model=k, **metrics(yte, fit_predict(k, Xtr, ytr, Xte))))

# ---------------- B. subject-independent split ----------------
tr = d.Subject.isin([f"S{i}" for i in range(2, 14)]).values
curves = {}
for k in ["RF", "CNN", "LR"]:
    pr = fit_predict(k, d.loc[tr, FEATS].values, y[tr], d.loc[~tr, FEATS].values)
    curves[k] = pr
    rows.append(dict(protocol="subject_indep_S2-13_vs_S14-17", model=k, **metrics(y[~tr], pr)))

# ---------------- C. LOSO ----------------
loso_rows = []
oof = {k: np.zeros(len(d)) for k in ["RF", "CNN", "LR"]}
for s in subjects:
    te = (d.Subject == s).values
    for k in ["RF", "CNN", "LR"]:
        pr = fit_predict(k, d.loc[~te, FEATS].values, y[~te], d.loc[te, FEATS].values)
        oof[k][te] = pr
        loso_rows.append(dict(subject=s, model=k, **metrics(y[te], pr)))
    print("LOSO done:", s, flush=True)
L = pd.DataFrame(loso_rows); L.to_csv(f"{OUT}/loso_per_subject.csv", index=False)
summ = L.groupby("model")[["acc", "f1", "auc", "ap"]].agg(["mean", "std"]).round(3)
summ.to_csv(f"{OUT}/loso_summary.csv"); print(summ)
for k in oof:
    rows.append(dict(protocol="LOSO_pooled", model=k, **metrics(y, oof[k])))

pd.DataFrame(rows).round(4).to_csv(f"{OUT}/main_results.csv", index=False)

# ---------------- D. ROC / PR curves (pooled LOSO predictions) ----------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for k, pr in oof.items():
    fpr, tpr, _ = roc_curve(y, pr); ax[0].plot(fpr, tpr, label=f"{k} (AUC={roc_auc_score(y, pr):.3f})")
    pcs, rec, _ = precision_recall_curve(y, pr); ax[1].plot(rec, pcs, label=f"{k} (AP={average_precision_score(y, pr):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (LOSO, pooled)")
ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall (LOSO, pooled)")
for a in ax: a.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/roc_pr_loso.png", dpi=200); plt.close()

# per-subject bar chart
piv = L.pivot(index="subject", columns="model", values="acc").loc[subjects]
piv.plot.bar(figsize=(11, 4.5)); plt.ylabel("LOSO accuracy"); plt.title("Per-subject accuracy (held-out subject)")
plt.axhline(0.5, color="k", ls=":", lw=.8); plt.tight_layout(); plt.savefig(f"{OUT}/per_subject_acc.png", dpi=200); plt.close()

# confusion matrices for pooled LOSO
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, k in zip(ax, oof):
    sns.heatmap(confusion_matrix(y, (oof[k] >= .5).astype(int)), annot=True, fmt="d", cmap="Blues", ax=a,
                xticklabels=["Relaxed", "Stressed"], yticklabels=["Relaxed", "Stressed"]); a.set_title(f"{k} (LOSO)")
plt.tight_layout(); plt.savefig(f"{OUT}/confusion_loso.png", dpi=200); plt.close()

# ---------------- E. bias / robustness checks ----------------
# E1: feature-set ablation under LOSO (RF)
abl = []
for name, cols in [("ECG+EDA", FEATS), ("ECG_only", ECG), ("EDA_only", EDA)]:
    for s in subjects:
        te = (d.Subject == s).values
        pr = fit_predict("RF", d.loc[~te, cols].values, y[~te], d.loc[te, cols].values)
        abl.append(dict(features=name, subject=s, **metrics(y[te], pr)))
A = pd.DataFrame(abl); A.to_csv(f"{OUT}/ablation_loso_rf.csv", index=False)
print(A.groupby("features")[["acc", "f1", "auc"]].agg(["mean", "std"]).round(3))

# E2: per-subject z-score normalisation (removes each person's own baseline).
# Caveat to state in the report: this uses the test subject's own unlabeled feature statistics.
Z = d.copy(); Z[FEATS] = d.groupby("Subject")[FEATS].transform(lambda x: (x - x.mean()) / x.std())
nrm = []
for k in ["RF", "CNN", "LR"]:
    for s in subjects:
        te = (Z.Subject == s).values
        pr = fit_predict(k, Z.loc[~te, FEATS].values, y[~te], Z.loc[te, FEATS].values)
        nrm.append(dict(model=k, subject=s, **metrics(y[te], pr)))
N = pd.DataFrame(nrm); N.to_csv(f"{OUT}/loso_per_subject_normalised.csv", index=False)
print(N.groupby("model")[["acc", "f1", "auc"]].agg(["mean", "std"]).round(3))

# E3: class balance and per-class error per subject (is the model biased toward 'Relaxed'?)
bal = d.groupby("Subject").State.value_counts().unstack().loc[subjects]
bal["stressed_share"] = bal.Stressed / (bal.Relaxed + bal.Stressed)
bal.to_csv(f"{OUT}/class_balance_per_subject.csv")
print("Done. See", OUT)


In [ ]:
!python phase4_evaluation.py WESAD_ML_Ready_Features.csv

Step 4: zip the results so you can download them.

In [ ]:
!zip -r phase4_outputs.zip phase4_outputs
files.download('phase4_outputs.zip')